# PA Recon Lab: reproducible GPU pilot
Compare unaugmented and augmented training on a fixed synthetic development split.
Enable GPU and Internet. This uses the repository's own model, CLI, checkpoint format and loss.
The 100-sample subset is a pipeline pilot. This notebook requires a CUDA GPU and stops if one is absent.
Data: https://doi.org/10.5281/zenodo.21785486 (CC BY 4.0).


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

project = Path('/kaggle/working/pa-recon-lab')
output = Path('/kaggle/working/pa-recon-results')
source_files = {'pa_recon/__init__.py': '"""Research code for uncertainty-aware sparse-view PAI reconstruction."""\n\n__version__ = "0.1.0"\n', 'pa_recon/cli.py': '"""Command-line tools for synthetic sinogram-to-image training and evaluation."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport h5py\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom .data import H5PairInfo, H5PairedImages, _as_chw, _read_sample, inspect_pair, load_group_splits\nfrom .model import SparseViewRecon, gaussian_l1_loss\n\n\ndef download_command(args: argparse.Namespace) -> None:\n    from .download import download_subset\n    download_subset(args.output)\n\n\ndef run_command(args: argparse.Namespace) -> None:\n    from .experiment import run_experiment\n    run_experiment(args.config, args.output, args.file, args.require_cuda)\n\n\ndef _print_info(info: H5PairInfo, path: str, sample_axis: int) -> None:\n    print(json.dumps({"file": str(path), "input_key": info.input_key,\n                      "target_key": info.target_key, "sample_count": info.sample_count,\n                      "input_shape": info.input_shape, "target_shape": info.target_shape,\n                      "sample_axis": sample_axis}, indent=2))\n\n\ndef inspect_command(args: argparse.Namespace) -> None:\n    info = inspect_pair(args.file, args.input_key, args.target_key, args.sample_axis)\n    _print_info(info, args.file, args.sample_axis)\n    with h5py.File(args.file, "r") as handle:\n        for key in (args.input_key, args.target_key):\n            sample = _read_sample(handle[key], 0, args.sample_axis)\n            finite = sample[np.isfinite(sample)]\n            print(f"{key}: sample0 shape={sample.shape} p01={np.percentile(finite, 1):.6g} "\n                  f"p50={np.percentile(finite, 50):.6g} p99={np.percentile(finite, 99):.6g}")\n\n\ndef make_splits_command(args: argparse.Namespace) -> None:\n    """Create deterministic splits without separating repeated source IDs."""\n    if (not math.isfinite(args.train_fraction) or not math.isfinite(args.validation_fraction)\n            or min(args.train_fraction, args.validation_fraction) <= 0\n            or args.train_fraction + args.validation_fraction >= 1):\n        raise ValueError("Train/validation fractions must be positive and sum to less than one")\n    info = inspect_pair(args.file, args.input_key, args.target_key, args.sample_axis)\n    with h5py.File(args.file, "r") as handle:\n        ids = np.asarray(handle[args.id_key]) if args.id_key in handle else np.arange(info.sample_count)\n    if ids.shape != (info.sample_count,):\n        raise ValueError(f"ID dataset must have shape ({info.sample_count},); got {ids.shape}")\n    id_strings = [str(v.item() if hasattr(v, "item") else v) for v in ids]\n    unique_ids = list(dict.fromkeys(id_strings))\n    order = np.random.default_rng(args.seed).permutation(len(unique_ids))\n    n_train = int(len(unique_ids) * args.train_fraction)\n    n_val = int(len(unique_ids) * args.validation_fraction)\n    if min(n_train, n_val, len(unique_ids) - n_train - n_val) < 1:\n        raise ValueError("Fractions must leave at least one source group in each split")\n    group_labels = {group: "test" for group in unique_ids}\n    group_labels.update({unique_ids[i]: "train" for i in order[:n_train]})\n    group_labels.update({unique_ids[i]: "validation" for i in order[n_train:n_train + n_val]})\n    labels = [group_labels[group] for group in id_strings]\n    output = Path(args.output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(("sample_index", "group_id", "split"))\n        for index, (sample_id, split) in enumerate(zip(ids, labels)):\n            writer.writerow((index, str(sample_id.item() if hasattr(sample_id, "item") else sample_id), split))\n    print(f"Wrote {info.sample_count} sample-level splits to {output}; seed={args.seed}")\n\n\ndef estimate_scales_command(args: argparse.Namespace) -> None:\n    """Estimate independent normalization scales from a bounded train-only sample."""\n    if args.max_samples < 1 or args.values_per_sample < 1 or not 0 < args.quantile <= 1:\n        raise ValueError("max-samples and values-per-sample must be positive; quantile must be in (0, 1]")\n    info = inspect_pair(args.file, args.input_key, args.target_key, args.sample_axis)\n    splits = load_group_splits(args.splits, info.sample_count)\n    rng = np.random.default_rng(args.seed)\n    indices = list(rng.permutation(splits["train"])[:args.max_samples])\n    values: dict[str, list[np.ndarray]] = {"input": [], "target": []}\n    with h5py.File(args.file, "r") as handle:\n        for index in indices:\n            for name, key in (("input", args.input_key), ("target", args.target_key)):\n                sample = np.asarray(_read_sample(handle[key], index, args.sample_axis), dtype=np.float32).ravel()\n                sample = np.abs(sample[np.isfinite(sample)])\n                if sample.size:\n                    take = min(args.values_per_sample, sample.size)\n                    values[name].append(sample[rng.choice(sample.size, take, replace=False)])\n    scales = {name: float(np.quantile(np.concatenate(parts), args.quantile)) for name, parts in values.items() if parts}\n    if len(scales) != 2 or any(not np.isfinite(v) or v <= 0 for v in scales.values()):\n        raise ValueError("Could not estimate positive finite scales from the training split")\n    result = {"input_scale": scales["input"], "target_scale": scales["target"],\n              "quantile": args.quantile, "train_samples_used": len(indices),\n              "values_per_sample": args.values_per_sample, "seed": args.seed,\n              "split_csv_sha256": hashlib.sha256(Path(args.splits).read_bytes()).hexdigest()}\n    output = Path(args.output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(result, indent=2))\n\n\ndef _corrupt_measurements(x: torch.Tensor, noise_std: float, dropout_probability: float,\n                         gain_jitter: float) -> torch.Tensor:\n    if gain_jitter:\n        gain = 1 + torch.empty((x.shape[0], 1, x.shape[-2], 1), device=x.device).uniform_(-gain_jitter, gain_jitter)\n        x = x * gain\n    if dropout_probability:\n        keep = (torch.rand((x.shape[0], 1, x.shape[-2], 1), device=x.device) >= dropout_probability)\n        x = x * keep\n    if noise_std:\n        x = x + torch.randn_like(x) * noise_std\n    return x\n\n\ndef _epoch(model: SparseViewRecon, loader: DataLoader, optimizer: torch.optim.Optimizer | None,\n           device: torch.device, augmentation: tuple[float, float, float] = (0.0, 0.0, 0.0),\n           mixup_alpha: float = 0.0) -> float:\n    training = optimizer is not None\n    model.train(training)\n    total, count = 0.0, 0\n    for x, y, _sample_indices in loader:\n        x, y = x.to(device), y.to(device)\n        if training:\n            noise_std, dropout_probability, gain_jitter = augmentation\n            if mixup_alpha > 0 and x.shape[0] > 1:\n                permutation = torch.randperm(x.shape[0], device=device)\n                beta = torch.distributions.Beta(mixup_alpha, mixup_alpha)\n                weights = beta.sample((x.shape[0],)).to(device=device, dtype=x.dtype).view(-1, 1, 1, 1)\n                x = weights * x + (1 - weights) * x[permutation]\n                y = weights * y + (1 - weights) * y[permutation]\n            x = _corrupt_measurements(x, noise_std, dropout_probability, gain_jitter)\n            optimizer.zero_grad(set_to_none=True)\n        with torch.set_grad_enabled(training):\n            mean, logvar = model(x, tuple(y.shape[-2:]))\n            loss = gaussian_l1_loss(mean, logvar, y)\n            if training:\n                loss.backward()\n                optimizer.step()\n        total += float(loss.detach()) * x.shape[0]\n        count += x.shape[0]\n    return total / max(count, 1)\n\n\ndef train_command(args: argparse.Namespace) -> None:\n    values = (args.noise_std, args.gain_jitter, args.mixup_alpha, args.learning_rate, args.detector_dropout)\n    if (not all(math.isfinite(v) for v in values) or min(args.noise_std, args.gain_jitter, args.mixup_alpha) < 0\n            or not 0 <= args.detector_dropout < 1 or args.learning_rate <= 0):\n        raise ValueError("Use finite nonnegative augmentation, dropout in [0, 1), and positive learning rate")\n    if min(args.epochs, args.batch_size, args.width, args.num_threads) < 1:\n        raise ValueError("epochs, batch-size, width and num-threads must be positive")\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    device = torch.device(args.device)\n    info = inspect_pair(args.file, args.input_key, args.target_key, args.sample_axis)\n    splits = load_group_splits(args.splits, info.sample_count)\n    print(f"Split sizes: { {name: len(values) for name, values in splits.items()} }")\n\n    datasets = {name: H5PairedImages(args.file, args.input_key, args.target_key,\n                                     indices, args.input_scale, args.target_scale, args.sample_axis)\n                for name, indices in splits.items()}\n    loaders = {name: DataLoader(dataset, batch_size=args.batch_size,\n                                shuffle=(name == "train"), num_workers=0)\n               for name, dataset in datasets.items()}\n    sample_x, sample_y, _ = datasets["train"][0]\n    if sample_x.shape[0] != sample_y.shape[0]:\n        raise ValueError("Input and target channel counts must match")\n    model = SparseViewRecon(in_channels=sample_x.shape[0], width=args.width).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=1e-4)\n    best_validation = float("inf")\n    output = Path(args.output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n\n    for epoch in range(1, args.epochs + 1):\n        epoch_start = time.perf_counter()\n        train_loss = _epoch(model, loaders["train"], optimizer, device,\n                            (args.noise_std, args.detector_dropout, args.gain_jitter), args.mixup_alpha)\n        validation_loss = _epoch(model, loaders["validation"], None, device)\n        if not math.isfinite(train_loss) or not math.isfinite(validation_loss):\n            raise FloatingPointError("Non-finite training/validation loss; no checkpoint saved for this epoch")\n        print(f"epoch={epoch:03d} train_nll_l1={train_loss:.6f} validation_nll_l1={validation_loss:.6f} "\n              f"seconds={time.perf_counter() - epoch_start:.2f}", flush=True)\n        if validation_loss < best_validation:\n            best_validation = validation_loss\n            checkpoint = {\n                "state_dict": model.state_dict(),\n                "model": {"name": "SparseViewRecon", "in_channels": sample_x.shape[0], "width": args.width,\n                          "output_size": list(sample_y.shape[-2:])},\n                "data": {"input_key": args.input_key, "target_key": args.target_key,\n                         "input_scale": args.input_scale, "target_scale": args.target_scale,\n                         "sample_axis": args.sample_axis},\n                "training": {"seed": args.seed, "epoch": epoch,\n                             "validation_loss": validation_loss,\n                             "batch_size": args.batch_size,\n                             "learning_rate": args.learning_rate,\n                             "augmentation": {"noise_std_normalized": args.noise_std,\n                                             "detector_dropout": args.detector_dropout,\n                                              "detector_gain_jitter": args.gain_jitter,\n                                              "linear_superposition_mixup_alpha": args.mixup_alpha},\n                             "max_epochs": args.epochs,\n                             "cpu_threads": args.num_threads,\n                             "torch_version": str(torch.__version__),\n                             "split_csv_sha256": hashlib.sha256(Path(args.splits).read_bytes()).hexdigest()},\n                "claim_boundary": "Synthetic detector-time sinogram to initial-pressure image; simulation-specific research baseline.",\n            }\n            temporary = output.with_name(output.name + ".tmp")\n            torch.save(checkpoint, temporary)\n            temporary.replace(output)\n    print(f"Best validation loss: {best_validation:.6f}; checkpoint: {output}")\n    for dataset in datasets.values():\n        dataset.close()\n\n\ndef evaluate_command(args: argparse.Namespace) -> None:\n    if args.test_noise_std < 0 or args.test_gain_jitter < 0 or not 0 <= args.test_detector_dropout < 1:\n        raise ValueError("test noise/gain jitter must be nonnegative and detector dropout must be in [0, 1)")\n    torch.manual_seed(args.corruption_seed)\n    device = torch.device(args.device)\n    checkpoint = torch.load(args.checkpoint, map_location=device, weights_only=True)\n    split_hash = hashlib.sha256(Path(args.splits).read_bytes()).hexdigest()\n    expected_hash = checkpoint.get("training", {}).get("split_csv_sha256")\n    if expected_hash is not None and expected_hash != split_hash:\n        raise ValueError("Evaluation split CSV differs from the checkpoint\'s training split manifest")\n    data_cfg = checkpoint["data"]\n    info = inspect_pair(args.file, data_cfg["input_key"], data_cfg["target_key"], data_cfg["sample_axis"])\n    splits = load_group_splits(args.splits, info.sample_count)\n    dataset = H5PairedImages(args.file, data_cfg["input_key"], data_cfg["target_key"],\n                             splits[args.split], data_cfg["input_scale"], data_cfg["target_scale"], data_cfg["sample_axis"])\n    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=0)\n    model_cfg = checkpoint["model"]\n    model = SparseViewRecon(model_cfg["in_channels"], model_cfg["width"]).to(device)\n    model.load_state_dict(checkpoint["state_dict"])\n    model.eval()\n    sample_groups: dict[int, str] = {}\n    with Path(args.splits).open("r", newline="", encoding="utf-8-sig") as stream:\n        for row in csv.DictReader(stream):\n            sample_groups[int(row["sample_index"])] = row["group_id"].strip()\n    accum = {key: 0.0 for key in ("model_mae", "zero_mae", "model_rmse_sq", "zero_rmse_sq", "target_sq", "target_count", "coverage_1sigma", "coverage_count", "uncertainty")}\n    per_sample: list[dict[str, float | int | str]] = []\n    with torch.no_grad():\n        for x, y, sample_indices in loader:\n            x, y = x.to(device), y.to(device)\n            x = _corrupt_measurements(x, args.test_noise_std, args.test_detector_dropout, args.test_gain_jitter)\n            mean, logvar = model(x, tuple(y.shape[-2:]))\n            error = mean - y\n            zero_error = y\n            sigma = torch.exp(0.5 * logvar)\n            count = y.numel()\n            accum["model_mae"] += error.abs().sum().item()\n            accum["zero_mae"] += zero_error.abs().sum().item()\n            accum["model_rmse_sq"] += error.square().sum().item()\n            accum["zero_rmse_sq"] += zero_error.square().sum().item()\n            accum["target_sq"] += y.square().sum().item()\n            accum["target_count"] += count\n            accum["coverage_1sigma"] += ((error.abs() <= sigma).float()).sum().item()\n            accum["coverage_count"] += count\n            accum["uncertainty"] += sigma.sum().item()\n            for item in range(x.shape[0]):\n                item_error = error[item]\n                item_zero_error = y[item]\n                item_target = y[item]\n                index = int(sample_indices[item])\n                per_sample.append({\n                    "sample_index": index,\n                    "group_id": sample_groups[index],\n                    "model_mae_normalized": item_error.abs().mean().item(),\n                    "model_rmse_normalized": item_error.square().mean().sqrt().item(),\n                    "zero_baseline_mae_normalized": item_zero_error.abs().mean().item(),\n                    "zero_baseline_rmse_normalized": item_zero_error.square().mean().sqrt().item(),\n                    "target_rms_normalized": item_target.square().mean().sqrt().item(),\n                    "one_sigma_coverage": (item_error.abs() <= sigma[item]).float().mean().item(),\n                    "mean_predicted_sigma_normalized": sigma[item].mean().item(),\n                })\n    n = accum["target_count"]\n    result = {\n        f"{args.split}_samples": len(dataset),\n        "evaluation_split": args.split,\n        "split_csv_sha256": split_hash,\n        "model_mae_normalized": accum["model_mae"] / n,\n        "zero_baseline_mae_normalized": accum["zero_mae"] / n,\n        "model_rmse_normalized": (accum["model_rmse_sq"] / n) ** 0.5,\n        "zero_baseline_rmse_normalized": (accum["zero_rmse_sq"] / n) ** 0.5,\n        "target_rms_normalized": (accum["target_sq"] / n) ** 0.5,\n        "predicted_1sigma_coverage": accum["coverage_1sigma"] / accum["coverage_count"],\n        "mean_predicted_sigma_normalized": accum["uncertainty"] / n,\n        "input_scale": data_cfg["input_scale"], "target_scale": data_cfg["target_scale"],\n        "test_corruption": {"noise_std_normalized": args.test_noise_std,\n                            "detector_dropout": args.test_detector_dropout,\n                            "detector_gain_jitter": args.test_gain_jitter,\n                            "seed": args.corruption_seed},\n        "reference": "synthetic initial pressure (IP) from digital-skin phantom dataset",\n    }\n    result["model_mae_target_units"] = result["model_mae_normalized"] * data_cfg["target_scale"]\n    result["zero_baseline_mae_target_units"] = result["zero_baseline_mae_normalized"] * data_cfg["target_scale"]\n    groups = sorted({str(row["group_id"]) for row in per_sample})\n    group_metrics = []\n    for group in groups:\n        rows = [row for row in per_sample if row["group_id"] == group]\n        group_metrics.append({\n            "group_id": group,\n            "samples": len(rows),\n            "model_mae_normalized": float(np.mean([row["model_mae_normalized"] for row in rows])),\n            "model_rmse_normalized": float(np.mean([row["model_rmse_normalized"] for row in rows])),\n            "zero_baseline_mae_normalized": float(np.mean([row["zero_baseline_mae_normalized"] for row in rows])),\n            "zero_baseline_rmse_normalized": float(np.mean([row["zero_baseline_rmse_normalized"] for row in rows])),\n            "one_sigma_coverage": float(np.mean([row["one_sigma_coverage"] for row in rows])),\n            "mean_predicted_sigma_normalized": float(np.mean([row["mean_predicted_sigma_normalized"] for row in rows])),\n        })\n    if len(group_metrics) >= 2:\n        rng = np.random.default_rng(20261011)\n        for metric in ("model_mae_normalized", "model_rmse_normalized", "zero_baseline_mae_normalized", "zero_baseline_rmse_normalized"):\n            values = np.asarray([row[metric] for row in group_metrics])\n            draws = rng.choice(values, size=(2000, len(values)), replace=True).mean(axis=1)\n            result[f"group_macro_{metric}"] = float(values.mean())\n            result[f"group_macro_{metric}_bootstrap_95ci"] = [float(v) for v in np.quantile(draws, [0.025, 0.975])]\n    result["group_count"] = len(group_metrics)\n    result["per_group_metrics"] = group_metrics\n    result["per_sample_metrics"] = per_sample\n    output = Path(args.output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(result, indent=2))\n    dataset.close()\n\n\ndef predict_command(args: argparse.Namespace) -> None:\n    device = torch.device(args.device)\n    checkpoint = torch.load(args.checkpoint, map_location=device, weights_only=True)\n    model_cfg, data_cfg = checkpoint["model"], checkpoint["data"]\n    with h5py.File(args.file, "r") as handle:\n        if data_cfg["input_key"] not in handle:\n            raise KeyError(f"Input HDF5 key missing: {data_cfg[\'input_key\']}")\n        raw_input = _read_sample(handle[data_cfg["input_key"]], args.sample_index, data_cfg["sample_axis"])\n    x_chw = _as_chw(raw_input, data_cfg["input_key"])\n    if not np.isfinite(x_chw).all():\n        raise ValueError(f"Non-finite values at sample {args.sample_index}")\n    x = torch.from_numpy(np.ascontiguousarray(x_chw / data_cfg["input_scale"]))\n    model = SparseViewRecon(model_cfg["in_channels"], model_cfg["width"]).to(device)\n    model.load_state_dict(checkpoint["state_dict"])\n    model.eval()\n    with torch.no_grad():\n        mean, logvar = model(x.unsqueeze(0).to(device), tuple(model_cfg["output_size"]))\n    Path(args.output).parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(args.output,\n                        sinogram=x.numpy() * data_cfg["input_scale"],\n                        reconstruction=(mean[0].cpu().numpy() * data_cfg["target_scale"]),\n                        predicted_sigma=(torch.exp(0.5 * logvar[0]).cpu().numpy() * data_cfg["target_scale"]),\n                        sample_index=args.sample_index,\n                        reference_type="synthetic IP phantom reference; not clinical ground truth")\n    print(f"Wrote reconstruction and predicted uncertainty to {args.output}")\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(prog="pa-recon", description="Research tools for sinogram-to-image PAI reconstruction")\n    sub = parser.add_subparsers(dest="command", required=True)\n    download_parser = sub.add_parser("download-data", help="Download and checksum-verify the public 100-sample subset")\n    download_parser.add_argument("--output", default="data/example_subset_100_samples.h5")\n    download_parser.set_defaults(function=download_command)\n    run_parser = sub.add_parser("run", help="Run a configured comparison with reproducibility manifests")\n    run_parser.add_argument("--config", default="configs/pilot.json")\n    run_parser.add_argument("--file", help="Override the dataset location declared in the config")\n    run_parser.add_argument("--output", default="runs/pilot")\n    run_parser.add_argument("--require-cuda", action="store_true")\n    run_parser.set_defaults(function=run_command)\n    split_parser = sub.add_parser("make-splits", help="Create deterministic sample-level synthetic-data splits")\n    split_parser.add_argument("--file", required=True)\n    split_parser.add_argument("--input-key", default="sinogram")\n    split_parser.add_argument("--target-key", default="IP")\n    split_parser.add_argument("--id-key", default="sample_id")\n    split_parser.add_argument("--sample-axis", type=int, default=0)\n    split_parser.add_argument("--seed", type=int, default=20261011)\n    split_parser.add_argument("--train-fraction", type=float, default=0.8)\n    split_parser.add_argument("--validation-fraction", type=float, default=0.1)\n    split_parser.add_argument("--output", required=True)\n    split_parser.set_defaults(function=make_splits_command)\n    scale_parser = sub.add_parser("estimate-scales", help="Estimate normalization from training samples only")\n    scale_parser.add_argument("--file", required=True)\n    scale_parser.add_argument("--input-key", default="sinogram")\n    scale_parser.add_argument("--target-key", default="IP")\n    scale_parser.add_argument("--sample-axis", type=int, default=0)\n    scale_parser.add_argument("--splits", required=True)\n    scale_parser.add_argument("--max-samples", type=int, default=512)\n    scale_parser.add_argument("--values-per-sample", type=int, default=4096)\n    scale_parser.add_argument("--quantile", type=float, default=0.999)\n    scale_parser.add_argument("--seed", type=int, default=20261011)\n    scale_parser.add_argument("--output", required=True)\n    scale_parser.set_defaults(function=estimate_scales_command)\n    for name, function in (("inspect", inspect_command), ("train", train_command),\n                           ("evaluate", evaluate_command), ("predict", predict_command)):\n        child = sub.add_parser(name)\n        child.add_argument("--file", required=True, help="Photoacoustic HDF5 dataset file")\n        child.set_defaults(function=function)\n        if name in ("train", "evaluate", "predict"):\n            child.add_argument("--num-threads", type=int, default=4,\n                               help="PyTorch CPU thread limit; set to 1 or 2 for small machines")\n        if name in ("inspect", "train"):\n            child.add_argument("--input-key", required=True, help="Input dataset key, e.g. sinogram")\n            child.add_argument("--target-key", required=True, help="Reference dataset key, e.g. IP")\n            child.add_argument("--sample-axis", type=int, default=0)\n        if name == "train":\n            child.add_argument("--splits", required=True, help="CSV: sample_index,group_id,split; synthetic samples use sample_id groups")\n            child.add_argument("--input-scale", type=float, required=True, help="Fixed sinogram scale from training data")\n            child.add_argument("--target-scale", type=float, required=True, help="Fixed IP scale from training data")\n            child.add_argument("--output", required=True, help="Checkpoint path (.pt)")\n            child.add_argument("--epochs", type=int, default=50)\n            child.add_argument("--batch-size", type=int, default=2)\n            child.add_argument("--width", type=int, default=16)\n            child.add_argument("--learning-rate", type=float, default=1e-3)\n            child.add_argument("--noise-std", type=float, default=0.01, help="Normalized Gaussian sinogram noise augmentation")\n            child.add_argument("--detector-dropout", type=float, default=0.05, help="Probability of dropping a detector per sample")\n            child.add_argument("--gain-jitter", type=float, default=0.1, help="Uniform per-detector gain perturbation")\n            child.add_argument("--mixup-alpha", type=float, default=0.0,\n                               help="Beta-distributed pair mixing for linear-acoustics superposition (0 disables)")\n            child.add_argument("--seed", type=int, default=20261011)\n            child.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n        elif name == "evaluate":\n            child.add_argument("--splits", required=True)\n            child.add_argument("--split", choices=("validation", "test"), default="test")\n            child.add_argument("--checkpoint", required=True)\n            child.add_argument("--output", required=True, help="Write aggregate held-out metrics to JSON")\n            child.add_argument("--batch-size", type=int, default=8)\n            child.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n            child.add_argument("--test-noise-std", type=float, default=0.0)\n            child.add_argument("--test-detector-dropout", type=float, default=0.0)\n            child.add_argument("--test-gain-jitter", type=float, default=0.0)\n            child.add_argument("--corruption-seed", type=int, default=20261011)\n        elif name == "predict":\n            child.add_argument("--checkpoint", required=True)\n            child.add_argument("--sample-index", type=int, required=True)\n            child.add_argument("--output", required=True, help="Compressed NPZ with reconstruction and uncertainty")\n            child.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    return parser\n\n\ndef main() -> None:\n    args = build_parser().parse_args()\n    if hasattr(args, "num_threads"):\n        if args.num_threads < 1:\n            raise ValueError("num-threads must be positive")\n        torch.set_num_threads(args.num_threads)\n    args.function(args)\n\n\nif __name__ == "__main__":\n    main()\n', 'pa_recon/data.py': '"""HDF5 input/target pairs and explicit sample-group split handling."""\n\nfrom __future__ import annotations\n\nimport csv\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Literal\n\nimport h5py\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\nSplit = Literal["train", "validation", "test"]\n\n\n@dataclass(frozen=True)\nclass H5PairInfo:\n    input_key: str\n    target_key: str\n    sample_count: int\n    input_shape: tuple[int, ...]\n    target_shape: tuple[int, ...]\n\n\ndef inspect_pair(path: str | Path, input_key: str, target_key: str, sample_axis: int = 0) -> H5PairInfo:\n    """Inspect paired datasets without loading their contents into memory."""\n    with h5py.File(path, "r") as handle:\n        missing = [key for key in (input_key, target_key) if key not in handle]\n        if missing:\n            available = sorted(handle.keys())\n            raise KeyError(f"Missing HDF5 key(s) {missing}. Top-level keys: {available}")\n        x, y = handle[input_key], handle[target_key]\n        if x.ndim < 3 or y.ndim < 3:\n            raise ValueError(f"Expected batched image arrays; got {input_key}={x.shape}, {target_key}={y.shape}")\n        axis_x = sample_axis % x.ndim\n        axis_y = sample_axis % y.ndim\n        if x.shape[axis_x] != y.shape[axis_y]:\n            raise ValueError(f"Pair sample counts differ: {x.shape} vs {y.shape}")\n        return H5PairInfo(input_key, target_key, int(x.shape[axis_x]), tuple(x.shape), tuple(y.shape))\n\n\ndef load_group_splits(path: str | Path, sample_count: int) -> dict[Split, list[int]]:\n    """Read a CSV with sample_index,group_id,split and enforce group isolation."""\n    grouped: dict[str, set[str]] = {}\n    splits: dict[Split, list[int]] = {"train": [], "validation": [], "test": []}\n    seen: set[int] = set()\n    with Path(path).open("r", newline="", encoding="utf-8-sig") as stream:\n        reader = csv.DictReader(stream)\n        required = {"sample_index", "group_id", "split"}\n        if not reader.fieldnames or not required.issubset(reader.fieldnames):\n            raise ValueError("Split CSV must contain columns: sample_index,group_id,split")\n        for row in reader:\n            index = int(row["sample_index"])\n            group = row["group_id"].strip()\n            split = row["split"].strip().lower()\n            if not group or split not in splits:\n                raise ValueError(f"Invalid group or split at sample {index}")\n            if index < 0 or index >= sample_count or index in seen:\n                raise ValueError(f"Invalid or duplicate sample_index {index}")\n            seen.add(index)\n            grouped.setdefault(group, set()).add(split)\n            splits[split].append(index)  # type: ignore[index]\n    if seen != set(range(sample_count)):\n        missing = sorted(set(range(sample_count)) - seen)\n        raise ValueError(f"Split CSV must map every sample exactly once; missing {len(missing)} samples")\n    leaking = [group for group, labels in grouped.items() if len(labels) != 1]\n    if leaking:\n        raise ValueError(f"Volunteer/session leakage across splits for group(s): {leaking[:8]}")\n    empty = [name for name, indices in splits.items() if not indices]\n    if empty:\n        raise ValueError(f"Each split must contain samples; empty: {empty}")\n    return splits\n\n\ndef _as_chw(array: np.ndarray, key: str) -> np.ndarray:\n    array = np.asarray(array, dtype=np.float32)\n    if array.ndim == 2:\n        return array[None, ...]\n    if array.ndim != 3:\n        raise ValueError(f"Each sample in {key!r} must be HxW, CxHxW, or HxWxC; got {array.shape}")\n    if array.shape[0] <= 4:\n        return array\n    if array.shape[-1] <= 4:\n        return np.moveaxis(array, -1, 0)\n    raise ValueError(f"Cannot infer channel axis for {key!r} sample shape {array.shape}")\n\n\ndef _read_sample(dataset: h5py.Dataset, index: int, sample_axis: int) -> np.ndarray:\n    axis = sample_axis % dataset.ndim\n    selection = [slice(None)] * dataset.ndim\n    selection[axis] = index\n    return np.asarray(dataset[tuple(selection)])\n\n\nclass H5PairedImages(Dataset[tuple[torch.Tensor, torch.Tensor, int]]):\n    """Read paired HDF5 arrays by sample index with a lazy per-process file handle."""\n\n    def __init__(self, path: str | Path, input_key: str, target_key: str,\n                 indices: list[int], input_scale: float, target_scale: float, sample_axis: int = 0):\n        if not np.isfinite(input_scale) or not np.isfinite(target_scale) or input_scale <= 0 or target_scale <= 0:\n            raise ValueError("input and target scales must be positive and fixed from training data")\n        self.path = str(path)\n        self.input_key = input_key\n        self.target_key = target_key\n        self.indices = list(indices)\n        self.input_scale = float(input_scale)\n        self.target_scale = float(target_scale)\n        self.sample_axis = sample_axis\n        self._handle: h5py.File | None = None\n\n    def _file(self) -> h5py.File:\n        if self._handle is None:\n            self._handle = h5py.File(self.path, "r")\n        return self._handle\n\n    def close(self) -> None:\n        if self._handle is not None:\n            self._handle.close()\n            self._handle = None\n\n    def __len__(self) -> int:\n        return len(self.indices)\n\n    def __getitem__(self, position: int) -> tuple[torch.Tensor, torch.Tensor, int]:\n        sample_index = self.indices[position]\n        handle = self._file()\n        x = _read_sample(handle[self.input_key], sample_index, self.sample_axis)\n        y = _read_sample(handle[self.target_key], sample_index, self.sample_axis)\n        x_chw = _as_chw(x, self.input_key)\n        y_chw = _as_chw(y, self.target_key)\n        if x_chw.shape[0] != y_chw.shape[0]:\n            raise ValueError(f"Input/target channel mismatch at sample {sample_index}: {x_chw.shape} vs {y_chw.shape}")\n        if not np.isfinite(x_chw).all() or not np.isfinite(y_chw).all():\n            raise ValueError(f"Non-finite values at sample {sample_index}")\n        return (torch.from_numpy(np.ascontiguousarray(x_chw / self.input_scale)),\n                torch.from_numpy(np.ascontiguousarray(y_chw / self.target_scale)), sample_index)\n', 'pa_recon/download.py': '"""Checksum-verified access to the public development subset."""\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\nimport urllib.request\n\nSUBSET_URL = "https://zenodo.org/records/21785486/files/example_subset_100_samples.h5?download=1"\nSUBSET_MD5 = "f9cab78d59b66eba53cc8a80bf6cc6a5"\n\n\ndef file_digest(path: str | Path, algorithm: str = "sha256") -> str:\n    digest = hashlib.new(algorithm)\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef download_subset(output: str | Path) -> Path:\n    path = Path(output)\n    if path.exists():\n        if file_digest(path, "md5") != SUBSET_MD5:\n            raise ValueError(f"Existing file checksum differs from the published subset: {path}")\n        print(f"Verified existing subset: {path}", flush=True)\n        return path\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".part")\n    if temporary.exists():\n        raise FileExistsError(f"Partial download exists; inspect or remove it before retrying: {temporary}")\n    stream = temporary.open("xb")\n    try:\n        request = urllib.request.Request(SUBSET_URL, headers={"User-Agent": "pa-recon-lab/0.1.0"})\n        with stream, urllib.request.urlopen(request, timeout=120) as response:\n            while block := response.read(1024 * 1024):\n                stream.write(block)\n        if file_digest(temporary, "md5") != SUBSET_MD5:\n            raise ValueError("Downloaded file does not match the dataset\'s published MD5")\n        temporary.replace(path)\n    except BaseException:\n        temporary.unlink(missing_ok=True)\n        raise\n    print(f"Downloaded and verified: {path}", flush=True)\n    return path\n', 'pa_recon/experiment.py': '"""Run a declared comparison through the package\'s canonical CLI functions."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nimport torch\n\nfrom .download import file_digest\n\n\ndef run_experiment(config_path: str, output: str, file_override: str | None = None,\n                   require_cuda: bool = False) -> None:\n    from .cli import build_parser\n\n    if require_cuda and not torch.cuda.is_available():\n        raise RuntimeError("A CUDA GPU was requested but is unavailable. Enable the lab\'s GPU accelerator.")\n    config = json.loads(Path(config_path).read_text(encoding="utf-8"))\n    if config.get("schema_version") != 1:\n        raise ValueError("Expected experiment schema_version 1")\n    arms = config["arms"]\n    names = [arm["name"] for arm in arms]\n    if not arms or len(names) != len(set(names)) or any(\n        not name or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in name) for name in names\n    ):\n        raise ValueError("Arms need unique lowercase names containing letters, digits, underscores or hyphens")\n    source = Path(file_override or config["data_file"]).resolve()\n    root = Path(output).resolve()\n    if root.exists() and any(root.iterdir()):\n        raise FileExistsError(f"Use a new output directory to preserve existing experiments: {root}")\n    root.mkdir(parents=True, exist_ok=True)\n    seed = config["seed"]\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    parser = build_parser()\n\n    def invoke(arguments: list[str]) -> None:\n        args = parser.parse_args(arguments)\n        if hasattr(args, "num_threads"):\n            torch.set_num_threads(args.num_threads)\n        args.function(args)\n\n    splits = root / "splits.csv"\n    scales_path = root / "scales.json"\n    invoke(["make-splits", "--file", str(source), "--output", str(splits), "--seed", str(seed)])\n    invoke(["estimate-scales", "--file", str(source), "--splits", str(splits), "--output", str(scales_path),\n            "--seed", str(seed)])\n    scales = json.loads(scales_path.read_text(encoding="utf-8"))\n    manifest = {\n        "config": config, "dataset_sha256": file_digest(source), "dataset_bytes": source.stat().st_size,\n        "split_sha256": file_digest(splits), "device": device,\n        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,\n        "python": sys.version, "platform": platform.platform(),\n        "torch": str(torch.__version__), "numpy": np.__version__,\n        "package_source_sha256": hashlib.sha256(b"".join(\n            p.name.encode() + p.read_bytes() for p in sorted(Path(__file__).parent.glob("*.py"))\n        )).hexdigest(),\n        "status": "running", "completed_arms": [],\n    }\n    manifest_path = root / "manifest.json"\n    manifest_path.write_text(json.dumps(manifest, indent=2) + "\\n", encoding="utf-8")\n    for arm in arms:\n        arm_root = root / arm["name"]\n        arm_root.mkdir()\n        checkpoint = arm_root / "model.pt"\n        parameters = {**config["training"], **arm.get("training", {})}\n        arguments = ["train", "--file", str(source), "--input-key", "sinogram", "--target-key", "IP",\n                     "--splits", str(splits), "--input-scale", str(scales["input_scale"]),\n                     "--target-scale", str(scales["target_scale"]), "--seed", str(seed),\n                     "--device", device, "--output", str(checkpoint)]\n        allowed = {"epochs", "batch_size", "width", "learning_rate", "noise_std", "detector_dropout",\n                   "gain_jitter", "mixup_alpha", "num_threads"}\n        if set(parameters) - allowed:\n            raise ValueError(f"Unknown training parameters: {sorted(set(parameters) - allowed)}")\n        for key, value in parameters.items():\n            arguments.extend(["--" + key.replace("_", "-"), str(value)])\n        invoke(arguments)\n        for split in ("validation", "test"):\n            invoke(["evaluate", "--file", str(source), "--splits", str(splits), "--checkpoint", str(checkpoint),\n                    "--split", split, "--device", device, "--output", str(arm_root / f"{split}_clean.json")])\n        for case in config.get("stress_cases", []):\n            case_name = case["name"]\n            if not case_name or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in case_name):\n                raise ValueError("Stress case names must be safe lowercase filenames")\n            invoke(["evaluate", "--file", str(source), "--splits", str(splits), "--checkpoint", str(checkpoint),\n                    "--device", device, "--output", str(arm_root / f"test_{case_name}.json"),\n                    "--test-noise-std", str(case["noise_std"]), "--test-detector-dropout", str(case["detector_dropout"]),\n                    "--test-gain-jitter", str(case["gain_jitter"]), "--corruption-seed", str(seed)])\n        manifest["completed_arms"].append(arm["name"])\n        manifest_path.write_text(json.dumps(manifest, indent=2) + "\\n", encoding="utf-8")\n    manifest["status"] = "complete"\n    manifest_path.write_text(json.dumps(manifest, indent=2) + "\\n", encoding="utf-8")\n    print(f"Experiment complete: {root}", flush=True)\n', 'pa_recon/model.py': '"""Residual U-Net with a mean reconstruction and per-pixel log variance."""\n\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        groups = lambda channels: next(g for g in range(min(8, channels), 0, -1) if channels % g == 0)\n        self.layers = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),\n            nn.GroupNorm(groups(out_channels), out_channels),\n            nn.GELU(),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),\n            nn.GroupNorm(groups(out_channels), out_channels),\n            nn.GELU(),\n        )\n        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.layers(x) + self.skip(x)\n\n\nclass SparseViewRecon(nn.Module):\n    """Residual U-Net mapping a 2-D measurement representation to an image."""\n\n    def __init__(self, in_channels: int = 1, width: int = 32):\n        super().__init__()\n        self.enc1 = ConvBlock(in_channels, width)\n        self.enc2 = ConvBlock(width, width * 2)\n        self.enc3 = ConvBlock(width * 2, width * 4)\n        self.pool = nn.AvgPool2d(2)\n        self.middle = ConvBlock(width * 4, width * 8)\n        self.up3 = nn.ConvTranspose2d(width * 8, width * 4, 2, stride=2)\n        self.dec3 = ConvBlock(width * 8, width * 4)\n        self.up2 = nn.ConvTranspose2d(width * 4, width * 2, 2, stride=2)\n        self.dec2 = ConvBlock(width * 4, width * 2)\n        self.up1 = nn.ConvTranspose2d(width * 2, width, 2, stride=2)\n        self.dec1 = ConvBlock(width * 2, width)\n        self.head = nn.Conv2d(width, in_channels * 2, 1)\n\n    def forward(self, x: torch.Tensor, output_size: tuple[int, int] | None = None) -> tuple[torch.Tensor, torch.Tensor]:\n        original_size = x.shape[-2:]\n        pad_h, pad_w = (-original_size[0]) % 8, (-original_size[1]) % 8\n        if pad_h or pad_w:\n            x = F.pad(x, (0, pad_w, 0, pad_h))\n        e1 = self.enc1(x)\n        e2 = self.enc2(self.pool(e1))\n        e3 = self.enc3(self.pool(e2))\n        z = self.middle(self.pool(e3))\n        d3 = self.dec3(torch.cat((self.up3(z), e3), dim=1))\n        d2 = self.dec2(torch.cat((self.up2(d3), e2), dim=1))\n        d1 = self.dec1(torch.cat((self.up1(d2), e1), dim=1))\n        if output_size is not None:\n            if tuple(output_size) != tuple(d1.shape[-2:]):\n                d1 = F.interpolate(d1, size=output_size, mode="bilinear", align_corners=False)\n        elif pad_h or pad_w:\n            d1 = d1[..., :original_size[0], :original_size[1]]\n        mean, log_variance = self.head(d1).chunk(2, dim=1)\n        return mean, log_variance.clamp(-8.0, 4.0)\n\n\ndef gaussian_l1_loss(mean: torch.Tensor, log_variance: torch.Tensor,\n                     target: torch.Tensor, l1_weight: float = 0.1) -> torch.Tensor:\n    """Heteroscedastic Gaussian NLL plus a small L1 term."""\n    nll = 0.5 * (torch.exp(-log_variance) * (mean - target).square() + log_variance)\n    return nll.mean() + l1_weight * (mean - target).abs().mean()\n', 'configs/pilot.json': '{\n  "schema_version": 1,\n  "data_file": "data/example_subset_100_samples.h5",\n  "seed": 20261011,\n  "training": {"epochs": 50, "batch_size": 4, "width": 16, "learning_rate": 0.001, "num_threads": 4},\n  "arms": [\n    {"name": "unaugmented", "training": {"noise_std": 0, "detector_dropout": 0, "gain_jitter": 0, "mixup_alpha": 0}},\n    {"name": "augmented", "training": {"noise_std": 0.01, "detector_dropout": 0.05, "gain_jitter": 0.05, "mixup_alpha": 0.4}}\n  ],\n  "stress_cases": [\n    {"name": "mild", "noise_std": 0.01, "detector_dropout": 0.05, "gain_jitter": 0.05},\n    {"name": "moderate", "noise_std": 0.025, "detector_dropout": 0.1, "gain_jitter": 0.1},\n    {"name": "severe", "noise_std": 0.05, "detector_dropout": 0.2, "gain_jitter": 0.2}\n  ]\n}\n', 'LICENSE': 'MIT License\n\nCopyright (c) 2026 PA Recon Lab contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'}
for relative, content in source_files.items():
    destination = project / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding='utf-8')
print('Embedded source SHA256: b7ef15ed6ab4fed6fc2091033541ae03ac13d6b6c2a85dc412de72239db10cbb')
environment = dict(os.environ, PYTHONPATH=str(project), PYTHONUNBUFFERED='1')


In [ ]:
subprocess.run([sys.executable, '-m', 'pa_recon.cli', 'download-data'],
               cwd=project, env=environment, check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pa_recon.cli', 'run',
                '--config', 'configs/pilot.json', '--output', str(output), '--require-cuda'],
               cwd=project, env=environment, check=True)
print('Download the pa-recon-results output directory: weights, metrics, splits, scales and manifest.')
